In [ ]:
# --- Xray Cloud: Count Test Runs in a Test Plan (PROJ-1003) ---
import requests, getpass, json, collections

PLAN_KEY = "PROJ-1003"

XRAY_AUTH_URL    = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_GRAPHQL_URL = "https://xray.cloud.getxray.app/api/v2/graphql"

# TEMP: bypass SSL certificate verification to unblock now
VERIFY_SSL = False
if not VERIFY_SSL:
    from urllib3.exceptions import InsecureRequestWarning
    import urllib3; urllib3.disable_warnings(InsecureRequestWarning)

# ---- prompt for proper Xray API keys (NOT Atlassian PAT) ----
CID  = getpass.getpass("XRAY_CLIENT_ID (Xray → API Keys): ")
CSEC = getpass.getpass("XRAY_CLIENT_SECRET (Xray → API Keys): ")

def auth():
    r = requests.post(XRAY_AUTH_URL, json={"client_id": CID, "client_secret": CSEC},
                      timeout=30, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"Auth failed: {r.status_code} {r.text[:300]}")
    return r.text.strip('"')

TOKEN   = auth()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

def gql(query, variables=None):
    r = requests.post(XRAY_GRAPHQL_URL, json={"query": query, "variables": variables or {}},
                      headers=HEADERS, timeout=60, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"GraphQL HTTP {r.status_code}: {r.text[:300]}")
    data = r.json()
    if "errors" in data:
        raise SystemExit(f"GraphQL errors: {data['errors']}")
    return data["data"]

# 1) Resolve Test Plan KEY -> issueId
Q_PLANS = """
query($jql:String!, $limit:Int!, $start:Int!) {
  getTestPlans(jql:$jql, limit:$limit, start:$start) {
    total start limit
    results { issueId jira(fields:["key","summary"]) }
  }
}
"""
plans = gql(Q_PLANS, {"jql": f'key = "{PLAN_KEY}"', "limit": 1, "start": 0})
if not plans["getTestPlans"]["results"]:
    raise SystemExit(f"No Test Plan found for key {PLAN_KEY}")
PLAN_ID = plans["getTestPlans"]["results"][0]["issueId"]
PLAN_NAME = f'{plans["getTestPlans"]["results"][0]["jira"]["key"]} - {plans["getTestPlans"]["results"][0]["jira"]["summary"]}'

# 2) Page through Test Executions in the plan to collect issueIds
Q_PLAN_EXECS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    testExecutions(limit:$limit, start:$start) {
      total start limit
      results { issueId }
    }
  }
}
"""
exec_ids = []
start = 0
page = 100
while True:
    d = gql(Q_PLAN_EXECS, {"planId": PLAN_ID, "limit": page, "start": start})
    te = d["getTestPlan"]["testExecutions"]
    exec_ids += [x["issueId"] for x in te["results"]]
    start += te["limit"]
    if start >= te["total"]:
        break

# 3) Pull Test Runs by batches and count statuses
Q_RUNS = """
query($execIds:[String], $limit:Int!, $start:Int!) {
  getTestRuns(testExecIssueIds:$execIds, limit:$limit, start:$start) {
    total start limit
    results { status { name } }
  }
}
"""

counts = collections.Counter()
total_runs = 0

if exec_ids:  # only query runs if we have executions
    BATCH = 50
    for i in range(0, len(exec_ids), BATCH):
        batch = exec_ids[i:i+BATCH]
        start = 0
        limit = 100
        while True:
            data = gql(Q_RUNS, {"execIds": batch, "limit": limit, "start": start})
            runs = data["getTestRuns"]
            for tr in runs["results"]:
                sname = (tr.get("status") or {}).get("name") or "UNKNOWN"
                counts[sname] += 1
                total_runs += 1
            start += runs["limit"]
            if start >= runs["total"]:
                break

# Map to canonical buckets 
PASSED   = sum(counts.get(k, 0) for k in ("PASSED", "Pass", "OK"))
FAILED   = sum(counts.get(k, 0) for k in ("FAILED", "Fail"))
DESCOPED = sum(counts.get(k, 0) for k in ("DESCOPED", "Not Executed (Descoped)", "SKIPPED (Descoped)"))
TODO     = sum(counts.get(k, 0) for k in ("TODO", "TO DO", "To Do", "NOT_RUN", "Untested"))

# Executed = tests that actually ran (passed/failed).
executed = PASSED + FAILED
pass_pct = round((PASSED / executed * 100.0), 2) if executed else 0.0

summary = {
    "Test Plan": PLAN_NAME,
    "Totals (Test Runs)": {
        "Total": total_runs,
        "Passed": PASSED,
        "Failed": FAILED,
        "Descoped": DESCOPED,
        "TO DO": TODO,
        "Pass Rate % (of executed)": pass_pct
    },
    "Other Statuses": {
        k: v for k, v in counts.items()
        if k not in {
            "PASSED", "Pass", "OK",
            "FAILED", "Fail",
            "DESCOPED", "Not Executed (Descoped)", "SKIPPED (Descoped)",
            "TODO", "TO DO", "To Do", "NOT_RUN", "Untested"
        }
    }
}

(json.dumps(summary, indent=2))

# Nice Markdown table to paste to Jira/Confluence/Slack
print("\n".join([
    f"### Test Plan Rollup — {PLAN_NAME}",
    "",
    "| Metric | Count |",
    "|---|---|",
    f"| Total | {summary['Totals (Test Runs)']['Total']} |",
    f"| Passed | {summary['Totals (Test Runs)']['Passed']} |",
    f"| Failed | {summary['Totals (Test Runs)']['Failed']} |",
    f"| Descoped | {summary['Totals (Test Runs)']['Descoped']} |",
    f"| To Do | {summary['Totals (Test Runs)']['TO DO']} |",
    f"| Pass Rate % (of executed) | {summary['Totals (Test Runs)']['Pass Rate % (of executed)']} |",
]))
